In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import joblib
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

Mounted at /content/drive


In [2]:
data_path = (
    "/content/drive/MyDrive/"
    "AutoSenseAI/data/duration/"
    "service_duration_benchmark.csv"
)

df = pd.read_csv(data_path)

print("Dataset:", df.shape)
print("Unique repair titles:", df["fix_title"].nunique())

Dataset: (314, 11)
Unique repair titles: 174


In [3]:
target = "est_labor_hours"

primary_features = [
    "fix_title",
    "difficulty_level",
    "probability_rank",
    "parts_cost_mid_usd",
    "parts_cost_range_usd"
]

fallback_features = [
    "fix_title",
    "difficulty_level",
    "probability_rank"
]

y = df[target]

In [4]:
primary_preprocessor = ColumnTransformer(
    transformers=[
        (
            "title_text",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1,
                max_features=1000
            ),
            "fix_title"
        ),

        (
            "difficulty",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            ["difficulty_level"]
        ),

        (
            "numeric",
            SimpleImputer(strategy="median"),
            [
                "probability_rank",
                "parts_cost_mid_usd",
                "parts_cost_range_usd"
            ]
        )
    ]
)

In [5]:
final_primary_model = Pipeline([
    (
        "preprocessor",
        primary_preprocessor
    ),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    )
])

final_primary_model.fit(
    df[primary_features],
    y
)

print("Primary model trained on all rows.")

Primary model trained on all rows.


In [6]:
fallback_preprocessor = ColumnTransformer(
    transformers=[
        (
            "title_text",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1,
                max_features=1000
            ),
            "fix_title"
        ),

        (
            "difficulty",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            ["difficulty_level"]
        ),

        (
            "numeric",
            SimpleImputer(strategy="median"),
            ["probability_rank"]
        )
    ]
)

final_fallback_model = Pipeline([
    (
        "preprocessor",
        fallback_preprocessor
    ),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    )
])

final_fallback_model.fit(
    df[fallback_features],
    y
)

print("Fallback model trained.")

Fallback model trained.


In [7]:
model_dir = (
    "/content/drive/MyDrive/"
    "AutoSenseAI/models"
)

os.makedirs(model_dir, exist_ok=True)

primary_path = os.path.join(
    model_dir,
    "service_duration_random_forest_final.pkl"
)

fallback_path = os.path.join(
    model_dir,
    "service_duration_random_forest_task_only_fallback.pkl"
)

joblib.dump(
    final_primary_model,
    primary_path
)

joblib.dump(
    final_fallback_model,
    fallback_path
)

print(primary_path)
print(fallback_path)

/content/drive/MyDrive/AutoSenseAI/models/service_duration_random_forest_final.pkl
/content/drive/MyDrive/AutoSenseAI/models/service_duration_random_forest_task_only_fallback.pkl


In [8]:
metadata = {
    "task": "service-duration-regression",
    "target": "estimated_labor_hours",

    "primary_model": {
        "algorithm": "RandomForestRegressor",
        "variant": "task_plus_cost",
        "features": primary_features,
        "cv_mean_mae_minutes": 13.976,
        "cv_mean_rmse_minutes": 23.696,
        "cv_mean_r2": 0.642
    },

    "fallback_model": {
        "algorithm": "RandomForestRegressor",
        "variant": "task_only",
        "features": fallback_features,
        "cv_mean_mae_minutes": 17.243,
        "cv_mean_rmse_minutes": 28.665,
        "cv_mean_r2": 0.554
    },

    "dataset_note": (
        "MechanicDB benchmark uses estimated labor hours, "
        "not observed real-world workshop completion times."
    )
}

metadata_path = os.path.join(
    model_dir,
    "service_duration_model_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved:", metadata_path)

Metadata saved: /content/drive/MyDrive/AutoSenseAI/models/service_duration_model_metadata.json


In [9]:
sample = df[primary_features].iloc[[0]]

prediction_hours = final_primary_model.predict(sample)[0]
prediction_minutes = prediction_hours * 60

print("Task:", sample["fix_title"].iloc[0])
print("Predicted hours:", round(prediction_hours, 2))
print("Predicted minutes:", round(prediction_minutes))
print("Reference hours:", df[target].iloc[0])

Task: Inspect wheel speed sensor connector and harness routing
Predicted hours: 0.4
Predicted minutes: 24
Reference hours: 0.4
